In [ ]:
!pip install -q pyroomacoustics soundfile scipy tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 34.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import numpy as np
import pandas as pd
import pyroomacoustics as pra
from tqdm import tqdm
import os

In [ ]:
np.random.seed(42)


In [ ]:
MATERIALS = {
    "painted_plaster": 0.07,
    "concrete": 0.03,
    "glass": 0.05,
    "wood": 0.20,
    "gypsum": 0.10,
    "carpet": 0.45,
    "curtain": 0.60,
    "acoustic_panel": 0.80
}

WALL_CHOICES  = ["painted_plaster", "concrete"]
FLOOR_CHOICES = ["wood", "carpet"]
CEIL_CHOICES  = ["painted_plaster"]



In [ ]:
def rt60_schroeder(rir, fs, t_start_db=-5, t_end_db=-35):
    rir = np.asarray(rir, dtype=float)
    rir = rir / (np.max(np.abs(rir)) + 1e-12)

    edc = np.cumsum(rir[::-1] ** 2)[::-1]
    edc /= np.max(edc) + 1e-12
    edc_db = 10 * np.log10(edc + 1e-12)

    idx_start = np.where(edc_db <= t_start_db)[0]
    idx_end   = np.where(edc_db <= t_end_db)[0]

    if len(idx_start) == 0 or len(idx_end) == 0:
        return None

    i1, i2 = idx_start[0], idx_end[0]
    if i2 <= i1 + 10:
        return None

    t = np.arange(len(rir)) / fs
    slope, _ = np.polyfit(t[i1:i2], edc_db[i1:i2], 1)

    if slope >= 0:
        return None

    return float(-60.0 / slope)



In [ ]:
def effective_absorption(L, W, H, wall_a, floor_a, ceil_a):
    wall_area = 2 * (L*H + W*H)
    floor_area = L * W
    ceil_area  = L * W
    total_area = wall_area + floor_area + ceil_area

    return (
        wall_a * wall_area +
        floor_a * floor_area +
        ceil_a * ceil_area
    ) / total_area


In [ ]:
def simulate_rt60(L, W, H, alpha_eff, fs=16000, max_order=20):
    room = pra.ShoeBox(
        [L, W, H],
        fs=fs,
        materials=pra.Material(float(alpha_eff)),
        max_order=max_order
    )

    margin = 0.7
    src = [
        np.random.uniform(margin, L - margin),
        np.random.uniform(margin, W - margin),
        np.random.uniform(1.2, min(1.7, H - 0.5))
    ]

    mic = [
        np.random.uniform(margin, L - margin),
        np.random.uniform(margin, W - margin),
        np.random.uniform(1.2, min(1.7, H - 0.5))
    ]

    room.add_source(src)
    room.add_microphone_array(
        pra.MicrophoneArray(np.array([mic]).T, fs)
    )

    room.compute_rir()
    return rt60_schroeder(room.rir[0][0], fs)




In [ ]:
def generate_dataset(n=1000, fs=16000, max_order=20):
    rows = []

    for _ in tqdm(range(n)):
        # Room dimensions (residential)
        L = np.random.uniform(3.0, 6.5)
        W = np.random.uniform(3.0, 5.5)
        H = np.random.uniform(2.4, 3.2)

        wall_mat  = np.random.choice(WALL_CHOICES)
        floor_mat = np.random.choice(FLOOR_CHOICES)
        ceil_mat  = np.random.choice(CEIL_CHOICES)

        wall_a  = MATERIALS[wall_mat]
        floor_a = MATERIALS[floor_mat]
        ceil_a  = MATERIALS[ceil_mat]

        # Panel coverage (strong enough to learn)
        panel_coverage = np.random.uniform(0.25, 0.7)
        panel_a = MATERIALS["acoustic_panel"]

        alpha_before = effective_absorption(L, W, H, wall_a, floor_a, ceil_a)

        treated_wall_a = wall_a * (1 - panel_coverage) + panel_a * panel_coverage
        alpha_after = effective_absorption(L, W, H, treated_wall_a, floor_a, ceil_a)

        rt60_b = simulate_rt60(L, W, H, alpha_before, fs, max_order)
        rt60_a = simulate_rt60(L, W, H, alpha_after,  fs, max_order)

        if rt60_b is None or rt60_a is None:
            continue

        rows.append({
            "L": L, "W": W, "H": H,
            "wall_mat": wall_mat,
            "floor_mat": floor_mat,
            "ceil_mat": ceil_mat,
            "wall_a": wall_a,
            "floor_a": floor_a,
            "ceil_a": ceil_a,
            "panel_coverage": panel_coverage,
            "alpha_eff_before": alpha_before,
            "alpha_eff_after": alpha_after,
            "rt60_before": rt60_b,
            "rt60_after": rt60_a,
            "rt60_delta": rt60_b - rt60_a
        })

    return pd.DataFrame(rows)



In [ ]:
df_test = generate_dataset(n=200)

print("Samples:", len(df_test))
print("Mean RT60 before:", df_test["rt60_before"].mean())
print("Mean RT60 after :", df_test["rt60_after"].mean())
print("Mean delta      :", df_test["rt60_delta"].mean())
print("Improvement rate:", (df_test["rt60_delta"] > 0).mean())

df_test.head()

100%|██████████| 200/200 [00:04<00:00, 45.03it/s]


Samples: 200
Mean RT60 before: 0.37827815067904624
Mean RT60 after : 0.3014223960773198
Mean delta      : 0.07685575460172649
Improvement rate: 0.93


,L,W,H,wall_mat,floor_mat,ceil_mat,wall_a,floor_a,ceil_a,panel_coverage,alpha_eff_before,alpha_eff_after,rt60_before,rt60_after,rt60_delta
0,4.310890,5.376786,2.985595,painted_plaster,wood,painted_plaster,0.07,0.20,0.07,0.320208,0.098917,0.228679,0.389712,0.383384,0.006328
1,4.836648,4.079863,2.632983,painted_plaster,wood,painted_plaster,0.07,0.20,0.07,0.312772,0.099684,0.223738,0.355528,0.339422,0.016106
2,6.379712,5.020993,2.643691,painted_plaster,carpet,painted_plaster,0.07,0.45,0.07,0.557905,0.167892,0.365328,0.488268,0.304901,0.183367
3,5.712965,5.348747,3.115862,concrete,wood,painted_plaster,0.03,0.20,0.07,0.664843,0.079343,0.350698,0.416796,0.326272,0.090524
4,3.260927,5.467217,3.017796,concrete,carpet,painted_plaster,0.03,0.45,0.07,0.252485,0.122838,0.238778,0.406616,0.376315,0.030301


In [ ]:
df = generate_dataset(n=6000)
print("Final dataset size:", df.shape)


100%|██████████| 6000/6000 [02:21<00:00, 42.37it/s]

Final dataset size: (6000, 15)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

os.makedirs("/content/drive/MyDrive/AcuVisor", exist_ok=True)

df.to_csv(
    "/content/drive/MyDrive/AcuVisor/synthetic_acoustics_dataset.csv",
    index=False
)

print("Dataset saved to Google Drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset saved to Google Drive


In [ ]:
print("DATASET SUMMARY")
print("---------------")
print("Samples:", len(df))
print("Mean RT60 before:", df["rt60_before"].mean())
print("Mean RT60 after :", df["rt60_after"].mean())
print("Mean RT60 delta :", df["rt60_delta"].mean())
print("Improvement rate:", (df["rt60_delta"] > 0).mean())


DATASET SUMMARY
---------------
Samples: 6000
Mean RT60 before: 0.3832512423073028
Mean RT60 after : 0.30278833843340236
Mean RT60 delta : 0.08046290387390041
Improvement rate: 0.9196666666666666


In [ ]:
df.to_csv("synthetic_acoustics_dataset.csv", index=False)
print("Saved synthetic_acoustics_dataset.csv")


Saved synthetic_acoustics_dataset.csv
